# 02 — Transfer Learning with YOLOv8 (rigorous walk-through)

> **Why this notebook exists.** A YOLO detector trained from scratch on
> a few thousand warehouse frames will hit ~25 % mAP and overfit fast.
> The same architecture *fine-tuned* from the COCO-pretrained checkpoint
> reaches > 0.85 mAP@0.5 in 50 epochs on the same hardware. This notebook
> walks through **why** that works, **how** to set it up correctly in
> Ultralytics, and **how to diagnose** the failure modes that look like
> "the model just doesn't detect anything".
>
> Every section maps to a section of the LOGIVISION soutenance. Run it
> top-to-bottom for the report; jump straight to §5 if you want the
> production-ready fine-tune recipe.

## What we'll cover

| § | Topic |
|---|---|
| 1 | Theory — what transfer learning *is*, and why pretrained backbones work |
| 2 | Setup — paths, MLflow, dataset assumptions |
| 3 | Inspect the dataset — class balance, bbox sizes, splits |
| 4 | **Baseline:** train YOLOv8n *from scratch* (random init) |
| 5 | **TL strategy A: feature extraction** — freeze backbone, train head only |
| 6 | **TL strategy B: full fine-tuning** — all layers trainable, layer-wise LR |
| 7 | **Custom head** — add Conv + Dense layers on top of YOLO features |
| 8 | Cross-run comparison in MLflow |
| 9 | Diagnose "precision = 0" — what to check before blaming the model |
| 10 | Promote the winner to MLflow Production |
| 11 | Reproduce on the Kaggle warehouse-delivery-box dataset |


## 1. Theory — what transfer learning is, and *why* it works on YOLO

### 1.1 The problem with training from scratch

A YOLOv8n has **3.0 M parameters** organised in:

- **Backbone (CSPDarknet)** — 9 stages of Conv + C2f modules. Learns
  low- to mid-level visual primitives: edges, corners, textures, parts
  of common objects.
- **Neck (PANet)** — 6 stages of upsample/concat/Conv. Aggregates
  multi-scale features.
- **Head (Detect)** — 3 detection branches that regress
  `(x, y, w, h, class_logits)` at strides 8/16/32.

To learn these primitives well, the network needs **on the order of
10⁵–10⁶ labelled images**. The warehouse dataset has ~10² — three orders
of magnitude too few. Training from scratch underfits catastrophically.

### 1.2 The transfer learning fix

The COCO-pretrained `yolov8n.pt` already has well-formed primitives in
its **backbone and neck**. Those primitives are *generic* — edges and
textures don't care whether the picture shows a giraffe (COCO) or a
pallet (warehouse). We **inherit those weights** and only adapt the
parts of the network that depend on the *task* (the head) and on the
*domain* (the late stages of the backbone, optionally).

Mathematically, given pretrained weights θ⁰ and a small target dataset
𝒟ₜ, we solve:

```
θ* = argmin_θ  L(θ; 𝒟ₜ)    starting from θ = θ⁰
```

rather than starting from a random θ. This biases optimisation toward
parameter regions that already encode useful visual features — a much
better initial point than random.

### 1.3 Two strategies

| Strategy | Frozen layers | Use when… |
|---|---|---|
| **A. Feature extraction** | Backbone (first ~10 layers) | Target domain is *very close* to source (COCO); tiny dataset; need fast convergence; risk of overfitting. |
| **B. Full fine-tuning** | None | Target domain differs (synthetic warehouses, security cams, …); medium-sized dataset; you have an hour+ of training compute. |

Empirically, on warehouse imagery, **B wins** by ~3-5 mAP points but takes
~2× longer to train. We'll run both below and compare in §8.

### 1.4 Layer-wise learning rates

When fine-tuning the *whole* network it's wise to apply a **lower LR**
to early layers (already well-tuned on COCO) and a **higher LR** to the
head (which must learn warehouse-specific classes). Ultralytics
exposes this implicitly via the `lr0` (head LR) and the `lrf` (final-to-
initial ratio) hyperparameters — it bakes the discriminative-LR pattern
of Howard & Ruder 2018 ("ULMFiT") into the training loop.


## 2. Setup

We expect:
- `datasets/processed/demo/data.yaml` populated by `dvc repro` (or by
  `make demo-data` if you haven't run DVC).
- `yolov8n.pt` at the repo root (auto-downloaded by Ultralytics on
  first use otherwise).
- MLflow stack running (`make up`) — UI on http://localhost:5050.

If anything is missing the cells below will raise clear errors — that's
intentional, we want failure diagnosis, not silent fallbacks.


In [ ]:
import sys, os, json, hashlib, time
from pathlib import Path
import warnings; warnings.filterwarnings('ignore', category=FutureWarning)

# Locate the repo root by walking up to the first directory with pyproject.toml.
REPO = Path.cwd().resolve()
while not (REPO / 'pyproject.toml').is_file() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
print('Repo root :', REPO)

DATA_YAML = REPO / 'datasets' / 'processed' / 'demo' / 'data.yaml'
assert DATA_YAML.is_file(), f'Run `make demo-data` first — missing {DATA_YAML}'

import yaml
data_cfg = yaml.safe_load(DATA_YAML.read_text())
CLASS_NAMES = data_cfg['names']
print(f'Classes ({len(CLASS_NAMES)}):', CLASS_NAMES)

# MLflow
import mlflow
TRACKING_URI = os.environ.get('MLFLOW_TRACKING_URI', 'http://localhost:5050')
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment('transfer-learning-walkthrough')
print('MLflow tracking URI :', TRACKING_URI)

## 3. Inspect the dataset — sanity before science

Before any training, the *single most informative* thing you can do is
count labels per class and per split. **An empty label directory is the
#1 cause of `precision = 0` in YOLO** (§9), and you find it by counting.


In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

stats = {}
for split in ('train', 'val', 'test'):
    img_dir = DATA_YAML.parent / 'images' / split
    lbl_dir = DATA_YAML.parent / 'labels' / split
    if not lbl_dir.is_dir():
        stats[split] = {'images': 0, 'labels': 0, 'class_counts': Counter()}
        continue
    counts = Counter()
    for txt in lbl_dir.rglob('*.txt'):
        for line in txt.read_text().splitlines():
            if line.strip():
                cls = int(line.split()[0])
                counts[CLASS_NAMES[cls] if cls < len(CLASS_NAMES) else f'#{cls}'] += 1
    stats[split] = {
        'images': len(list(img_dir.glob('*'))) if img_dir.is_dir() else 0,
        'labels': len(list(lbl_dir.rglob('*.txt'))),
        'class_counts': counts,
    }

print(f"{'split':<7} {'images':>8} {'lbl-files':>10}  per-class instances")
for split, s in stats.items():
    print(f'{split:<7} {s["images"]:>8} {s["labels"]:>10}  {dict(s["class_counts"])}')

# Sanity assertion: every split must have labels — empty splits silently
# break Ultralytics' validation (precision = 0 with no warning).
for split, s in stats.items():
    if s['images'] > 0:
        assert s['labels'] > 0, f'{split} has images but no labels — fix prepare_dataset.py'

In [ ]:
# Bounding-box size distribution per split. Tiny boxes (< 10 px on a
# 640-input image) are the second-most-common cause of poor recall.
import numpy as np

areas = {split: [] for split in stats}
for split in stats:
    lbl_dir = DATA_YAML.parent / 'labels' / split
    if not lbl_dir.is_dir():
        continue
    for txt in lbl_dir.rglob('*.txt'):
        for line in txt.read_text().splitlines():
            parts = line.split()
            if len(parts) >= 5:
                _, _, _, w, h = parts[:5]
                # YOLO labels are normalized 0..1; area is also normalized.
                areas[split].append(float(w) * float(h))

fig, ax = plt.subplots(figsize=(10, 3.5))
for split, vals in areas.items():
    if vals:
        ax.hist(np.array(vals) * 100, bins=30, alpha=0.55, label=f'{split} (n={len(vals)})')
ax.set_xlabel('Normalised bbox area (% of image)'); ax.set_ylabel('count')
ax.set_title('Bounding-box size distribution per split'); ax.legend(); plt.tight_layout(); plt.show()

## 4. Baseline — training YOLOv8n *from scratch*

This is the **negative control**. We initialise the network with random
weights (loading the *architecture* from `yolov8n.yaml`, NOT the
checkpoint) and train 1 epoch. The point is **not** to get a good model,
it's to give §5–6 something to beat.

Why 1 epoch? Because training a 3 M-parameter network from scratch to
convergence needs days on a CPU. The mAP after 1 epoch on a small
warehouse split is < 5 % — which makes the case for transfer learning
visually obvious in §8.


In [ ]:
from ultralytics import YOLO

baseline_dir = REPO / 'ml' / 'runs' / 'nb02_baseline_scratch'

with mlflow.start_run(run_name='baseline_from_scratch') as run:
    mlflow.set_tags({
        'strategy': 'from_scratch',
        'init': 'random (yolov8n.yaml)',
        'frozen_layers': 'none',
    })
    mlflow.log_params({'epochs': 1, 'imgsz': 320, 'batch': 8, 'lr0': 0.01})

    # NOTE: we load the .yaml (architecture only) NOT the .pt (weights+arch).
    scratch_model = YOLO('yolov8n.yaml')
    scratch_results = scratch_model.train(
        data=str(DATA_YAML),
        epochs=1, imgsz=320, batch=8,
        project=str(REPO / 'ml' / 'runs'),
        name='nb02_baseline_scratch',
        exist_ok=True, verbose=False, plots=False,
    )
    val = scratch_model.val(data=str(DATA_YAML), verbose=False)
    baseline_metrics = {
        'val_map50': float(val.box.map50),
        'val_map50_95': float(val.box.map),
        'val_precision': float(val.box.mp),
        'val_recall': float(val.box.mr),
    }
    mlflow.log_metrics(baseline_metrics)
    print('Baseline (from scratch) :', baseline_metrics)

## 5. Transfer learning A — *feature extraction* (freeze backbone)

We load the **COCO checkpoint** (`yolov8n.pt`) and freeze the first 10
layers. This means the **backbone weights are frozen**: gradients still
flow through them (PyTorch always computes grads on the forward graph)
but the optimizer never updates them — they keep their COCO values.

**What we'd expect to see:**
- Training loss drops fast in the first 1-2 epochs (only the head learns).
- Validation mAP plateaus at a "decent but not great" level — typically
  60–75 % of full fine-tuning, because the backbone never adapts.
- Training is ~30 % faster per epoch than full fine-tuning.

Hyperparameter rationale:
- `lr0 = 0.005` — slightly lower than the from-scratch default (0.01)
  because we don't want the head to forget COCO-aligned outputs too fast.
- `freeze = 10` — Ultralytics treats this as "freeze layers 0..9 of
  `model.model[*]`". You can verify with `model.model.named_parameters`
  (see the diagnostic cell below).


In [ ]:
# Verify what `freeze=10` actually freezes — a 2-line diagnostic that
# saves hours of "why didn't my freeze work" debugging.
tmp = YOLO('yolov8n.pt')
n_total = sum(p.numel() for p in tmp.model.parameters())
print(f'Total params           : {n_total:,}')
# Mimic what Ultralytics does: it sets `requires_grad=False` on layer indices < freeze.
for i, m in enumerate(tmp.model.model):
    state = '🧊 frozen' if i < 10 else '🔥 trainable'
    n_params = sum(p.numel() for p in m.parameters())
    print(f'  layer[{i:>2}] {state:<11} {type(m).__name__:<14} {n_params:>10,} params')

In [ ]:
with mlflow.start_run(run_name='tl_A_feature_extract') as run:
    mlflow.set_tags({
        'strategy': 'transfer_learning_A',
        'init': 'yolov8n.pt (COCO)',
        'frozen_layers': '0..9 (backbone)',
    })
    mlflow.log_params({'epochs': 3, 'imgsz': 320, 'batch': 8, 'lr0': 0.005, 'freeze': 10})

    tlA_model = YOLO('yolov8n.pt')
    tlA_model.train(
        data=str(DATA_YAML),
        epochs=3, imgsz=320, batch=8, lr0=0.005, freeze=10,
        project=str(REPO / 'ml' / 'runs'),
        name='nb02_tl_A_freeze_backbone',
        exist_ok=True, verbose=False, plots=False,
    )
    val_A = tlA_model.val(data=str(DATA_YAML), verbose=False)
    tlA_metrics = {
        'val_map50': float(val_A.box.map50),
        'val_map50_95': float(val_A.box.map),
        'val_precision': float(val_A.box.mp),
        'val_recall': float(val_A.box.mr),
    }
    mlflow.log_metrics(tlA_metrics)
    print('TL-A (feature extraction) :', tlA_metrics)

## 6. Transfer learning B — *full fine-tuning* (all layers trainable)

Same starting weights (`yolov8n.pt`) but **no freezing**. Every parameter
updates. We compensate for the higher risk of catastrophic forgetting by:

1. Starting with a small `lr0 = 0.001`.
2. Using `cos_lr=True` (cosine learning-rate schedule) so the LR anneals
   smoothly to zero — this is the Ultralytics default and matches the
   ULMFiT discriminative-LR recipe.
3. Adding `patience=20` early-stopping on the validation mAP plateau.

**Expected outcome:** higher mAP than §5, longer training per epoch.
This is the configuration that produced the `8a4db…` run in the repo
(mAP@0.5 = 0.86 after 50 epochs).


In [ ]:
with mlflow.start_run(run_name='tl_B_full_finetune') as run:
    mlflow.set_tags({
        'strategy': 'transfer_learning_B',
        'init': 'yolov8n.pt (COCO)',
        'frozen_layers': 'none — full fine-tune',
    })
    mlflow.log_params({'epochs': 3, 'imgsz': 320, 'batch': 8, 'lr0': 0.001, 'cos_lr': True})

    tlB_model = YOLO('yolov8n.pt')
    tlB_model.train(
        data=str(DATA_YAML),
        epochs=3, imgsz=320, batch=8, lr0=0.001, cos_lr=True,
        project=str(REPO / 'ml' / 'runs'),
        name='nb02_tl_B_full_finetune',
        exist_ok=True, verbose=False, plots=False,
    )
    val_B = tlB_model.val(data=str(DATA_YAML), verbose=False)
    tlB_metrics = {
        'val_map50': float(val_B.box.map50),
        'val_map50_95': float(val_B.box.map),
        'val_precision': float(val_B.box.mp),
        'val_recall': float(val_B.box.mr),
    }
    mlflow.log_metrics(tlB_metrics)
    print('TL-B (full fine-tune) :', tlB_metrics)

## 7. Custom head — adding Conv + Dense layers on top of YOLO features

The default YOLO Detect head outputs `(x, y, w, h, class_logits)` per
anchor. For some use-cases (e.g. predicting a *scalar* like "carton
fullness" or routing decisions) we want a different output entirely. We
demonstrate how to **bolt a custom Conv → AdaptiveAvgPool → Dense head**
onto the YOLO backbone, treating YOLO as a frozen feature extractor.

The point of this section is pedagogical: showing the *separation* of
the backbone (frozen, reused) and the task-specific head (trainable,
your own design). We don't train it end-to-end in this notebook because
it would need a *different* task and dataset than detection.


In [ ]:
import torch
import torch.nn as nn

# Load the COCO checkpoint and grab the backbone (everything before the
# Detect head). Ultralytics' DetectionModel exposes layers via .model.model[i].
yolo = YOLO('yolov8n.pt')
backbone_layers = list(yolo.model.model.children())[:-1]
print(f'Backbone composed of {len(backbone_layers)} blocks (Detect head excluded)')

class WarehouseClassifierHead(nn.Module):
    """Custom head: Conv → BN → ReLU → AdaptiveAvgPool → Dense (×2) → softmax.

    Pedagogical demo of how to layer your own classification head on top
    of a YOLO-derived feature extractor.

    Args:
        in_channels: channel count of the last backbone feature map (256
                     for yolov8n at stride 32).
        n_classes  : how many task-specific classes you want to predict.
    """
    def __init__(self, in_channels: int = 256, n_classes: int = 4):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)         # → (B, 64, 1, 1)
        self.dense = nn.Sequential(
            nn.Flatten(),                            # → (B, 64)
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(32, n_classes),                # logits — apply softmax in loss
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dense(self.pool(self.conv(x)))


# Quick forward-pass demo with a dummy input image.
demo_in = torch.randn(1, 3, 320, 320)
with torch.no_grad():
    feats = demo_in
    for layer in backbone_layers:
        # YOLO backbone modules expect a flat tensor (not a list) for early stages.
        feats = layer(feats) if not isinstance(feats, list) else feats[-1]
    if isinstance(feats, list):
        feats = feats[-1]
    print('Backbone output shape :', feats.shape)
    head = WarehouseClassifierHead(in_channels=feats.shape[1], n_classes=4)
    print('Head output shape     :', head(feats).shape)

print('\n→ Wire this head into a `nn.Sequential(backbone, head)` and train\n'
      '  with cross-entropy when you have labels for your scalar task\n'
      '  (e.g. "is this aisle full / half / empty / unknown").')

## 8. Cross-run comparison — point this at the jury

The three runs above all land in the same MLflow experiment
(`transfer-learning-walkthrough`). We pull them programmatically and
plot the metric side-by-side — this is the chart you want on the
soutenance slide.


In [ ]:
import pandas as pd
client = mlflow.tracking.MlflowClient()
exp = client.get_experiment_by_name('transfer-learning-walkthrough')
runs = client.search_runs([exp.experiment_id], order_by=['attribute.start_time ASC'])

records = []
for r in runs:
    records.append({
        'run_name': r.data.tags.get('mlflow.runName', r.info.run_id[:8]),
        'strategy': r.data.tags.get('strategy', '?'),
        'frozen':   r.data.tags.get('frozen_layers', '?'),
        **{k: round(v, 3) for k, v in r.data.metrics.items() if 'val_' in k},
    })
df = pd.DataFrame(records); display(df)

ax = df.set_index('run_name')[['val_map50', 'val_map50_95', 'val_precision', 'val_recall']].plot.bar(
    figsize=(10, 4), rot=12, color=['#2563EB', '#06B6D4', '#10B981', '#F59E0B'])
ax.set_ylabel('metric'); ax.set_title('Transfer-learning strategies — head-to-head')
ax.legend(loc='upper left', framealpha=0.9); plt.tight_layout(); plt.show()

## 9. Diagnostic — when `precision = 0` or stays near zero

If §5 or §6 produces `precision ≈ 0` after 3 epochs, the *model* is
almost never the problem. Run these checks **in order** — they're the
top-5 root causes I've seen on this codebase:

1. **Empty labels directory for a split** — `len(list(labels_dir.glob('*.txt'))) == 0`
   means YOLO sees zero ground-truth boxes for val/test → mAP is
   undefined and reported as 0. Fix in `ml/scripts/import_annotations.py`.
2. **Class-id mismatch** — the first integer on each label line must
   be < `len(data_cfg['names'])`. Ultralytics silently drops bad rows.
3. **Image-extension mismatch** — Ultralytics pairs `images/X.jpg` with
   `labels/X.txt`. If your images are `.png` and labels were exported
   for `.jpg`, no pair matches.
4. **Single-class collapse** — `single_cls: true` in data.yaml combined
   with multi-class labels collapses everything to class 0.
5. **Confidence threshold too high at val time** — for a *cold* model
   the default 0.25 is brutal. Drop to 0.10 to see if the model has
   *anything* useful: `model.val(conf=0.10)`.


In [ ]:
# Run the cheap version of every check above on the current split.
problems = []
for split in ('train', 'val', 'test'):
    img_dir = DATA_YAML.parent / 'images' / split
    lbl_dir = DATA_YAML.parent / 'labels' / split
    if not img_dir.is_dir(): continue
    images = {p.stem for p in img_dir.iterdir()}
    labels = {p.stem for p in lbl_dir.iterdir()} if lbl_dir.is_dir() else set()
    matched = images & labels
    unmatched_img = images - labels
    if not labels:                       problems.append(f'{split}: 0 label files')
    if len(unmatched_img) > len(matched): problems.append(f'{split}: {len(unmatched_img)} images have no labels')
    bad_class_ids = 0
    for txt in lbl_dir.rglob('*.txt') if lbl_dir.is_dir() else []:
        for line in txt.read_text().splitlines():
            if line.strip() and int(line.split()[0]) >= len(CLASS_NAMES):
                bad_class_ids += 1
    if bad_class_ids:                    problems.append(f'{split}: {bad_class_ids} rows have class_id >= len(names)')
if not problems:
    print('✓ No obvious data issues. If precision is still 0 after 3 epochs, lower conf to 0.10.')
else:
    print('Issues found:'); [print(' -', p) for p in problems]

## 10. Promote the winner to MLflow Production

Whichever run wins §8 (usually `tl_B_full_finetune`) gets promoted via
`ml/scripts/promote_model.py`. The thresholds in
`ml/configs/promotion_thresholds.yaml` (`map50 ≥ 0.65`, `map50_95 ≥ 0.40`,
`recall ≥ 0.55`) are the gate — runs below the line stay in `None`.

```bash
# From a terminal, not the notebook:
uv run python -m ml.scripts.promote_model --run-id <id_from_§8>      # → Staging
uv run python -m ml.scripts.promote_model --run-id <id> --approve    # → Production
```

The `inference_worker` picks up the new model on its next restart via
`resolve_model_weights('logivision-detector', stage='Production')`.


## 11. Reproduce on the Kaggle dataset

The demo dataset is small enough that the metric differences in §8 are
noisy. To get publication-grade numbers, pull the **Kaggle warehouse-
delivery-box** dataset and re-run §5 + §6 against it.

```bash
# One-time: configure Kaggle CLI
export KAGGLE_USERNAME=<your-username>
export KAGGLE_KEY=<your-api-key>
mkdir -p ~/.kaggle
python3 -c "import json,os; json.dump({'username':os.environ['KAGGLE_USERNAME'],'key':os.environ['KAGGLE_KEY']}, open(os.path.expanduser('~/.kaggle/kaggle.json'),'w'))"
chmod 600 ~/.kaggle/kaggle.json

# Pull the dataset (~860 MB)
uv run kaggle datasets download -d zoya77/warehouse-delivery-box-detection-dataset   -p data/raw/kaggle_box_detection/ --unzip

# Convert to YOLO splits, then point data.yaml at the new path
uv run python -m ml.scripts.import_annotations   --input data/raw/kaggle_box_detection/   --output datasets/processed/kaggle_v1/

# Update the config and re-train
sed -i.bak 's|datasets/processed/demo|datasets/processed/kaggle_v1|' ml/configs/yolov8n.yaml
make train
```

Then re-run §8 — the metric gap between TL-A and TL-B widens
substantially when the dataset is large enough for proper fine-tuning.


---

## TL;DR for the soutenance

> *"Nous fine-tunons YOLOv8n à partir des poids COCO. Deux stratégies
> sont comparées : (A) gel du backbone, entraînement de la tête seule,
> et (B) fine-tuning intégral avec un schedule learning-rate cosinus.
> Sur notre jeu de données warehouse, la stratégie B atteint mAP@0.5 =
> 0.86 en 50 epochs sur CPU. Les trois runs (scratch / TL-A / TL-B) sont
> tracés dans MLflow et le meilleur est promu en `Production` via le
> script `promote_model.py` qui vérifie automatiquement les seuils de
> qualité avant publication."*

Tags pour l'écrit : Transfer Learning · ULMFiT layer-wise LR · MLflow
Model Registry · Ultralytics `freeze` · Ablation study.
